# U.S. Recession Probability Model — 12-Month Ahead

A **probit regression** using NBER recession dates as the binary dependent variable,
driven by economic indicators across eight categories.

Based on the methodology of Estrella & Mishkin (1996, 1998), as used by the
NY Fed and Cleveland Fed.

**Framework:** P(Recession\_{t+12} = 1 | X\_t) = Φ(α₀ + α₁X\_t)

Where Φ(·) is the standard normal CDF and X\_t is a vector of economic indicators.

### Key design choices (all configurable below):
1. **Dependent variable**: "recession at month t+12" vs. "any recession in next 12 months"
2. **Feature selection**: data-driven via BIC, not hardcoded
3. **Estimation**: expanding-window pseudo out-of-sample
4. **Data universe**: all 41 FRED series across eight macroeconomic categories


In [ ]:
# Install dependencies
!pip install fredapi statsmodels scikit-learn matplotlib pandas numpy scipy -q

In [ ]:
# ============================================================
# FRED API Key Setup
# Uses Colab Secrets if available, otherwise prompts for input
# To set up: Colab sidebar > Secrets > Add FRED_API_KEY
# Get your free key at: https://fred.stlouisfed.org/docs/api/fred/
# ============================================================
try:
    from google.colab import userdata
    FRED_API_KEY = userdata.get("FRED_API_KEY")
    print("FRED API key loaded from Colab Secrets.")
except (ImportError, ModuleNotFoundError):
    # Not running in Colab
    from getpass import getpass
    FRED_API_KEY = getpass("Enter your FRED API key: ")
except Exception:
    # Colab but secret not set
    from getpass import getpass
    FRED_API_KEY = getpass("FRED_API_KEY secret not found. Enter your key: ")

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from fredapi import Fred
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

fred = Fred(api_key=FRED_API_KEY)
print("FRED connection established.")

## Configuration

All modeling choices are controlled here. Nothing downstream is hardcoded.

In [ ]:
# ============================================================
# MODEL CONFIGURATION — adjust these to explore alternatives
# ============================================================

# Dependent variable definition:
#   "point"  = recession specifically at month t+12 (NY Fed approach)
#   "window" = any recession during months t+1 through t+12
TARGET_DEFINITION = "point"

# Observation start date (CFNAI begins 1967)
OBS_START = "1967-01-01"

# Minimum expanding-window training size (months)
MIN_WINDOW = 120  # 10 years

# Maximum number of features for BIC selection
MAX_FEATURES_BIC = 9

# Threshold levels for probability interpretation
THRESHOLD_WARNING = 30   # "warrants attention"
THRESHOLD_ELEVATED = 50  # "more likely than not"

print(f"Target definition:  {TARGET_DEFINITION}")
print(f"Observation start:  {OBS_START}")
print(f"Min training window: {MIN_WINDOW} months")
print(f"Max BIC features:   {MAX_FEATURES_BIC}")

## Stage 1: Data Pipeline — All 41 FRED Series

The full indicator universe across eight macroeconomic categories.
The first series in each category is the strongest predictor per academic evidence.

In [ ]:
# ============================================================
# All 41 FRED series across 8 categories
# ============================================================
SERIES_CONFIG = {
    # --- National Economic Activity ---
    "CFNAI":    {"name": "Chicago Fed National Activity Index",   "category": "National Activity", "transform": "level"},
    "CFNAIMA3": {"name": "CFNAI 3-Month Moving Average",         "category": "National Activity", "transform": "level"},
    "GDPC1":    {"name": "Real GDP",                              "category": "National Activity", "transform": "yoy", "freq": "Q"},
    "USSLIND":  {"name": "Leading Index for the US",              "category": "National Activity", "transform": "level"},

    # --- Industrial Indicators ---
    "INDPRO":   {"name": "Industrial Production Index",           "category": "Industrial", "transform": "yoy"},
    "BSCICP02USM460S": {"name": "OECD Manufacturing Confidence",        "category": "Industrial", "transform": "level"},
    "TCU":      {"name": "Capacity Utilization",                  "category": "Industrial", "transform": "level"},
    "DGORDER":  {"name": "Durable Goods Orders",                  "category": "Industrial", "transform": "yoy"},
    "IPMAN":    {"name": "Industrial Production: Manufacturing",  "category": "Industrial", "transform": "yoy"},

    # --- Consumer Measures ---
    "UMCSENT":  {"name": "U. Michigan Consumer Sentiment",        "category": "Consumer", "transform": "level"},
    "PCECC96":  {"name": "Real Personal Consumption Expenditures","category": "Consumer", "transform": "yoy"},
    "DSPIC96":  {"name": "Real Disposable Personal Income",       "category": "Consumer", "transform": "yoy"},
    "RSAFS":    {"name": "Advance Retail Sales",                  "category": "Consumer", "transform": "yoy"},

    # --- Labor Market ---
    "UNRATE":   {"name": "Unemployment Rate",                     "category": "Labor", "transform": "level"},
    "ICSA":     {"name": "Initial Unemployment Claims",           "category": "Labor", "transform": "yoy", "freq": "W"},
    "PAYEMS":   {"name": "Total Nonfarm Payrolls",                "category": "Labor", "transform": "yoy"},
    "CIVPART":  {"name": "Labor Force Participation Rate",        "category": "Labor", "transform": "level"},
    "JTSJOL":   {"name": "Job Openings (JOLTS)",                  "category": "Labor", "transform": "yoy"},

    # --- Inflation ---
    "CPIAUCSL": {"name": "CPI All Urban Consumers",              "category": "Inflation", "transform": "yoy"},
    "PCEPILFE": {"name": "Core PCE Price Index",                  "category": "Inflation", "transform": "yoy"},
    "PCEPI":    {"name": "PCE Chain-Type Price Index",            "category": "Inflation", "transform": "yoy"},
    "CPILFESL": {"name": "Core CPI",                              "category": "Inflation", "transform": "yoy"},
    "PPIACO":   {"name": "PPI All Commodities",                   "category": "Inflation", "transform": "yoy"},

    # --- Housing ---
    "HOUST":    {"name": "Housing Starts",                        "category": "Housing", "transform": "yoy"},
    "PERMIT":   {"name": "Building Permits",                      "category": "Housing", "transform": "yoy"},
    "HSN1F":    {"name": "New One-Family Houses Sold",            "category": "Housing", "transform": "yoy"},
    "CSUSHPISA":{"name": "Case-Shiller National Home Price Index","category": "Housing", "transform": "yoy"},

    # --- Banking / Credit ---
    "BAA10YM":  {"name": "Baa Corp Bond - 10Y Treasury Spread",  "category": "Banking", "transform": "level"},
    "BUSLOANS": {"name": "Commercial & Industrial Loans",         "category": "Banking", "transform": "yoy"},
    "DRALACBS": {"name": "Delinquency Rate, All Loans",           "category": "Banking", "transform": "level", "freq": "Q"},
    "DRTSCILM": {"name": "Tightening Standards C&I Loans",        "category": "Banking", "transform": "level", "freq": "Q"},

    # --- Government Bond Yields ---
    "T10Y3M":   {"name": "10Y-3M Treasury Spread",               "category": "Yields", "transform": "level", "freq": "D"},
    "T10Y2Y":   {"name": "10Y-2Y Treasury Spread",               "category": "Yields", "transform": "level", "freq": "D"},
    "GS10":     {"name": "10-Year Treasury Yield",                "category": "Yields", "transform": "level"},
    "TB3MS":    {"name": "3-Month Treasury Bill Rate",            "category": "Yields", "transform": "level"},
    "FEDFUNDS": {"name": "Federal Funds Rate",                    "category": "Yields", "transform": "level"},
}

# Target variable (separate — not a feature)
TARGET_SERIES = {
    "USREC":         {"name": "NBER Recession Indicator",         "category": "Target"},
    "RECPROUSM156N": {"name": "Chauvet-Piger Recession Prob",     "category": "Benchmark"},
}

print(f"Data universe: {len(SERIES_CONFIG)} indicator series + {len(TARGET_SERIES)} target series")
print(f"Categories: {sorted(set(v['category'] for v in SERIES_CONFIG.values()))}")

In [ ]:
# ============================================================
# Fetch all series from FRED (with retry for transient errors)
# ============================================================
import time

OBS_START = "1967-01-01"  # CFNAI starts 1967

raw_data = pd.DataFrame()
failed = []

all_series = {**SERIES_CONFIG, **TARGET_SERIES}
MAX_RETRIES = 4

for sid, info in all_series.items():
    success = False
    for attempt in range(MAX_RETRIES):
        try:
            s = fred.get_series(sid, observation_start=OBS_START)

            # Resample non-monthly frequencies to monthly
            freq = info.get("freq", "M")
            if freq == "W":
                s = s.resample("MS").mean()
            elif freq == "D":
                s = s.resample("MS").last()
            elif freq == "Q":
                # Forward-fill quarterly to monthly
                s = s.resample("MS").ffill()

            raw_data[sid] = s
            print(f"  + {sid:12s} [{info.get('freq','M'):>1s}] {info['name']}")
            success = True
            break
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait = 2 ** (attempt + 1)  # 2s, 4s, 8s
                print(f"  ~ {sid:12s}       Retry {attempt+1}/{MAX_RETRIES-1} in {wait}s ({e})")
                time.sleep(wait)
            else:
                failed.append(sid)
                print(f"  x {sid:12s}       FAILED after {MAX_RETRIES} attempts: {e}")

# Ensure monthly alignment
raw_data.index = pd.to_datetime(raw_data.index)
raw_data = raw_data.resample("MS").last()

print(f"\nFetched {len(all_series) - len(failed)}/{len(all_series)} series.")
print(f"Date range: {raw_data.index.min().strftime('%Y-%m')} to {raw_data.index.max().strftime('%Y-%m')}")
if failed:
    print(f"Failed: {failed}")
    print("Note: FRED API can have transient errors. Re-run this cell to retry.")
raw_data.tail(3)

## Stage 2: Feature Engineering

Transformations are driven by the `transform` field in `SERIES_CONFIG`:
- `"yoy"` → 12-month percent change (for non-stationary level series)
- `"level"` → used as-is (for stationary/bounded series like spreads, rates, indexes)

Additionally constructs:
- **SPREAD** = GS10 − TB3MS (manual yield curve spread for pre-1982 history)
- **UNRATE_CHG3** = Sahm-style 3-month MA change in unemployment
- **Both dependent variable definitions** (configurable via `TARGET_DEFINITION`)

In [ ]:
# ============================================================
# Feature Engineering — transform-driven
# ============================================================
data = raw_data.copy()

# --- Construct derived spread (GS10 - TB3MS) for full history ---
if "GS10" in data.columns and "TB3MS" in data.columns:
    data["SPREAD"] = data["GS10"] - data["TB3MS"]

# --- Sahm-style unemployment rate change ---
if "UNRATE" in data.columns:
    unrate_ma3 = data["UNRATE"].rolling(3).mean()
    data["UNRATE_CHG3"] = unrate_ma3 - unrate_ma3.shift(12)

# --- Apply transforms from SERIES_CONFIG ---
feature_cols = []

for sid, info in SERIES_CONFIG.items():
    if sid not in data.columns:
        continue

    transform = info["transform"]

    if transform == "yoy":
        col_name = f"{sid}_YOY"
        data[col_name] = data[sid].pct_change(12) * 100
        feature_cols.append(col_name)
    elif transform == "level":
        feature_cols.append(sid)

# Add the derived features
if "SPREAD" in data.columns:
    feature_cols.append("SPREAD")
if "UNRATE_CHG3" in data.columns:
    feature_cols.append("UNRATE_CHG3")

# De-duplicate (SPREAD components GS10/TB3MS already in as levels)
feature_cols = sorted(set(feature_cols))

# --- Dependent variable: configurable ---
if TARGET_DEFINITION == "point":
    data["TARGET"] = data["USREC"].shift(-12)
    target_desc = "Recession specifically at month t+12"
elif TARGET_DEFINITION == "window":
    data["TARGET"] = data["USREC"].rolling(window=12).max().shift(-12)
    target_desc = "Any recession during months t+1 through t+12"
else:
    raise ValueError(f"Unknown TARGET_DEFINITION: {TARGET_DEFINITION}")

print(f"Target definition: {target_desc}")
print(f"Candidate features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

In [ ]:
# ============================================================
# Build model-ready dataframe
# ============================================================
# Problem: short-history series (e.g. CSUSHPISA from 1987, DRALACBS from 1985,
# JTSJOL from 2000) shrink the dataset dramatically when we dropna() across
# all features. With only 200-odd observations and 2-3 recessions, the model
# risks complete separation (perfect but meaningless fit).
#
# Solution: only include features that have data for at least 80% of the
# target period, preserving a large enough sample with enough recessions.

MIN_COVERAGE = 0.80  # feature must have data for 80% of the target period

# Map features back to categories (needed for exclusion reporting below)
feat_to_cat = {}
for sid, info in SERIES_CONFIG.items():
    cat = info["category"]
    if info["transform"] == "yoy":
        feat_to_cat[f"{sid}_YOY"] = cat
    else:
        feat_to_cat[sid] = cat
feat_to_cat["SPREAD"] = "Yields (derived)"
feat_to_cat["UNRATE_CHG3"] = "Labor (derived)"

# First, build the target column and determine the full date range
target_series = data["TARGET"].dropna()
date_range = target_series.index

# Filter features by coverage
available_features = []
excluded_features = []

for c in feature_cols:
    if c not in data.columns:
        continue
    coverage = data.loc[date_range, c].notna().mean()
    if coverage >= MIN_COVERAGE:
        available_features.append(c)
    else:
        excluded_features.append((c, coverage))

if excluded_features:
    print(f"Excluded {len(excluded_features)} features due to insufficient history (< {MIN_COVERAGE*100:.0f}% coverage):")
    for feat, cov in excluded_features:
        cat = feat_to_cat.get(feat, "?")
        print(f"  {feat:<20s} [{cat}] — {cov*100:.0f}% coverage")
    print()

# Now build model_df with only the well-covered features
model_df = data[available_features + ["TARGET", "USREC"]].dropna()

n_rec = int(model_df["TARGET"].sum())
pct_rec = model_df["TARGET"].mean() * 100

print(f"Model dataset: {len(model_df)} observations")
print(f"Date range:    {model_df.index.min().strftime('%Y-%m')} to {model_df.index.max().strftime('%Y-%m')}")
print(f"Recession obs: {n_rec} ({pct_rec:.1f}%)")
print(f"Features:      {len(available_features)} (of {len(feature_cols)} candidates)")



print("\nFeatures by category:")
for cat in sorted(set(feat_to_cat.values())):
    feats = [f for f in available_features if feat_to_cat.get(f) == cat]
    if feats:
        print(f"  {cat}: {feats}")

## Exploratory Data Analysis

Visualize key indicators from each category against NBER recession periods.

In [ ]:
# ============================================================
# EDA: One indicator per category vs. recessions
# ============================================================
# Pick the top indicator per category for visualization
eda_picks = []
category_order = ["Yields", "Yields (derived)", "Banking", "National Activity",
                  "Industrial", "Consumer", "Labor", "Labor (derived)",
                  "Inflation", "Housing"]
seen_cats = set()
for cat in category_order:
    for f in available_features:
        if feat_to_cat.get(f) == cat and cat not in seen_cats:
            eda_picks.append((f, cat))
            seen_cats.add(cat)
            break

n_plots = min(len(eda_picks), 10)
rows = (n_plots + 1) // 2
fig, axes = plt.subplots(rows, 2, figsize=(16, 3.2 * rows), sharex=True)
axes = axes.flat

usrec = data["USREC"].dropna()

for i, (col, cat) in enumerate(eda_picks[:n_plots]):
    ax = axes[i]
    if col in data.columns:
        s = data[col].dropna()
        ax.plot(s.index, s.values, linewidth=0.9, color="#1f77b4")
    ax.fill_between(usrec.index, ax.get_ylim()[0], ax.get_ylim()[1],
                    where=usrec.values == 1, color="gray", alpha=0.2)
    ax.set_title(f"{col}  [{cat}]", fontsize=10)
    ax.grid(True, alpha=0.15)

# Hide unused subplots
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Key Indicators by Category vs. NBER Recessions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Stage 3: Data-Driven Feature Selection via BIC

Instead of hardcoding features, we use **forward stepwise selection by BIC**
(Bayesian Information Criterion) to let the data determine which indicators
enter the model.

BIC penalizes model complexity more heavily than AIC, favoring parsimony —
consistent with Berge (2014)'s finding that "at the 12-month horizon,
parsimony dominates."

The selection proceeds:
1. Start with SPREAD (the single strongest predictor per Estrella & Mishkin)
2. Greedily add the feature that produces the largest BIC improvement
3. Stop when no addition improves BIC, or `MAX_FEATURES_BIC` is reached

In [ ]:
# ============================================================
# Univariate screening: rank all features by individual BIC
# ============================================================
y = model_df["TARGET"].astype(float)

univariate_results = []

for feat in available_features:
    X = sm.add_constant(model_df[[feat]].astype(float))
    try:
        res = sm.Probit(y, X).fit(disp=False, method="bfgs", maxiter=300)
        # Flag separation
        separated = (res.prsquared > 0.99 or
                     any(abs(res.params) > 100) or
                     any(np.isnan(res.bse)))
        univariate_results.append({
            "feature": feat,
            "category": feat_to_cat.get(feat, "?"),
            "bic": res.bic,
            "aic": res.aic,
            "pseudo_r2": res.prsquared,
            "coef": res.params.iloc[-1],
            "pvalue": res.pvalues.iloc[-1],
            "separated": separated,
        })
    except Exception as e:
        print(f"  Skip {feat}: {e}")

uni_df = pd.DataFrame(univariate_results).sort_values("bic")
n_sep = uni_df["separated"].sum()
if n_sep > 0:
    print(f"WARNING: {n_sep} feature(s) show complete separation (marked with *)\n")
print("Univariate BIC ranking (lower = better):\n")
display_df = uni_df.copy()
display_df["flag"] = uni_df["separated"].map({True: " *", False: ""})
print(display_df[["feature","category","bic","pseudo_r2","pvalue","flag"]].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# ============================================================
# Forward stepwise selection by BIC (with separation detection)
# ============================================================
def _has_separation(res):
    # Check if a probit result shows complete separation:
    # huge coefficients, nan standard errors, or pseudo R2 near 1
    if res.prsquared > 0.99:
        return True
    if any(abs(res.params) > 100):
        return True
    if any(np.isnan(res.bse)):
        return True
    return False


def forward_stepwise_bic(y, X_all, feature_names, max_features, seed_features=None):
    # Greedy forward selection by BIC.
    # Stops when BIC stops improving, max_features reached,
    # or adding a feature causes complete separation.
    selected = list(seed_features) if seed_features else []
    remaining = [f for f in feature_names if f not in selected]

    if selected:
        X_curr = sm.add_constant(X_all[selected].astype(float))
        seed_res = sm.Probit(y, X_curr).fit(disp=False, method="bfgs", maxiter=300)
        best_bic = seed_res.bic
    else:
        X_curr = sm.add_constant(pd.DataFrame(index=X_all.index))
        seed_res = sm.Probit(y, X_curr).fit(disp=False, method="bfgs", maxiter=300)
        best_bic = seed_res.bic

    history = [{"step": 0, "added": "/".join(selected) if selected else "(none)",
                "bic": best_bic, "n_features": len(selected)}]

    while remaining and len(selected) < max_features:
        candidates = []
        for feat in remaining:
            try_feats = selected + [feat]
            X_try = sm.add_constant(X_all[try_feats].astype(float))
            try:
                res = sm.Probit(y, X_try).fit(disp=False, method="bfgs", maxiter=300)
                if _has_separation(res):
                    print(f"    Skip {feat}: complete separation detected")
                    continue
                candidates.append((feat, res.bic))
            except Exception:
                pass

        if not candidates:
            break

        best_feat, best_candidate_bic = min(candidates, key=lambda x: x[1])

        if best_candidate_bic >= best_bic:
            break

        selected.append(best_feat)
        remaining.remove(best_feat)
        best_bic = best_candidate_bic

        history.append({"step": len(selected), "added": best_feat,
                        "bic": best_bic, "n_features": len(selected)})
        print(f"  Step {len(selected):2d}: +{best_feat:<18s} BIC={best_bic:.2f}")

    return selected, history


# Determine seed: use SPREAD if available
seed = ["SPREAD"] if "SPREAD" in available_features else []

print(f"Seed features: {seed if seed else '(none)'}")
print(f"Candidate pool: {len(available_features)} features")
print(f"Max features: {MAX_FEATURES_BIC}\n")

bic_selected, bic_history = forward_stepwise_bic(
    y=y,
    X_all=model_df[available_features],
    feature_names=available_features,
    max_features=MAX_FEATURES_BIC,
    seed_features=seed,
)

print(f"\nBIC-selected features ({len(bic_selected)}):")
for i, f in enumerate(bic_selected):
    cat = feat_to_cat.get(f, "?")
    print(f"  {i+1}. {f:<20s} [{cat}]")

In [ ]:
# ============================================================
# BIC selection path visualization
# ============================================================
hist_df = pd.DataFrame(bic_history)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_df["n_features"], hist_df["bic"], "o-", color="#1f77b4", linewidth=2, markersize=8)
for _, row in hist_df.iterrows():
    ax.annotate(row["added"], (row["n_features"], row["bic"]),
                textcoords="offset points", xytext=(8, 8), fontsize=8, rotation=20)
ax.set_xlabel("Number of Features")
ax.set_ylabel("BIC (lower = better)")
ax.set_title("Forward Stepwise BIC Selection Path", fontweight="bold")
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

## Stage 4: Probit Model Estimation

Four models of increasing complexity:
1. **NY Fed baseline**: Yield curve spread only (Estrella & Mishkin 1998)
2. **Wright extension**: Spread + Federal Funds Rate level (Wright 2006)
3. **BIC-selected**: Data-driven feature set from stepwise selection
4. **Full candidate set**: All available features (likely overfits — included for comparison)

All use `statsmodels.Probit` with BFGS optimization.

In [ ]:
# ============================================================
# Define model specifications
# ============================================================

# Build Wright features from available data
wright_features = []
if "SPREAD" in available_features:
    wright_features.append("SPREAD")
elif "T10Y3M" in available_features:
    wright_features.append("T10Y3M")
if "FEDFUNDS" in available_features:
    wright_features.append("FEDFUNDS")

# NY Fed: spread only
nyfed_features = [wright_features[0]] if wright_features else [available_features[0]]

models_spec = {
    "NY Fed (Spread Only)": nyfed_features,
    "Wright (Spread + FF)": wright_features,
    "BIC-Selected":         bic_selected,
    "Full Candidate Set":   available_features,
}

results = {}

for name, features in models_spec.items():
    feats = [f for f in features if f in model_df.columns]
    if not feats:
        print(f"  {name}: no valid features, skipping")
        continue

    X = sm.add_constant(model_df[feats].astype(float))
    try:
        res = sm.Probit(y, X).fit(disp=False, method="bfgs", maxiter=500)
        fitted = res.predict(X)
        results[name] = {
            "model": res,
            "features": feats,
            "fitted": fitted,
            "pseudo_r2": res.prsquared,
            "bic": res.bic,
            "aic": res.aic,
        }
        print(f"{'='*60}")
        print(f"  {name}  ({len(feats)} features)")
        print(f"{'='*60}")
        print(f"  Pseudo R²:  {res.prsquared:.4f}")
        print(f"  Log-Lik:    {res.llf:.1f}")
        print(f"  AIC:        {res.aic:.1f}")
        print(f"  BIC:        {res.bic:.1f}")
        print(f"  Features:   {feats}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

# --- Generate predictions through the latest available data ---
# model_df ends 12 months early because TARGET requires future data.
# For prediction, we only need features — not the target.
predict_df = data[available_features].dropna()
print(f"\nPrediction range: {predict_df.index.min().strftime('%Y-%m')} to {predict_df.index.max().strftime('%Y-%m')}")
print(f"  (model_df ends at {model_df.index.max().strftime('%Y-%m')} due to 12-month target lead)")
print(f"  (predict_df extends {len(predict_df) - len(model_df)} months further with current readings)")

# Re-predict fitted probabilities on the full predict_df range
for name, res_dict in results.items():
    feats = res_dict["features"]
    avail_pred = predict_df[feats].dropna()
    X_pred = sm.add_constant(avail_pred.astype(float))
    try:
        full_fitted = res_dict["model"].predict(X_pred)
        res_dict["fitted_full"] = full_fitted
    except Exception:
        res_dict["fitted_full"] = res_dict["fitted"]

print("\n" + "="*60)
print("  Model estimation complete.")

In [ ]:
# ============================================================
# BIC-selected model — full coefficient table
# ============================================================
if "BIC-Selected" in results:
    print("BIC-Selected Model Summary:\n")
    print(results["BIC-Selected"]["model"].summary())
else:
    print("BIC-Selected model not available.")

## Stage 4b: Expanding-Window Out-of-Sample Estimation

Each month's probability is generated from a model trained **only on data
available at that point**. This is the honest evaluation — no lookahead bias.

Uses the BIC-selected features (data-driven, not hardcoded).

In [ ]:
# ============================================================
# Expanding-window pseudo out-of-sample — BIC-selected model
# ============================================================
oos_features = bic_selected  # data-driven, not hardcoded
oos_probs = pd.Series(index=model_df.index, dtype=float)

n = len(model_df)
total_iters = n - MIN_WINDOW
print(f"Running expanding-window OOS estimation ({total_iters} iterations)...")
print(f"Features: {oos_features}\n")

first_error_shown = False
n_errors = 0

for i in range(MIN_WINDOW, n):
    train = model_df.iloc[:i]
    y_t = train["TARGET"].astype(float)
    X_t = sm.add_constant(train[oos_features].astype(float))

    try:
        res = sm.Probit(y_t, X_t).fit(disp=False, method="bfgs", maxiter=300)

        # Predict for current observation
        # Use manual prediction to avoid add_constant issues on single rows
        x_curr = model_df[oos_features].iloc[i].astype(float).values
        x_with_const = np.concatenate([[1.0], x_curr])  # prepend intercept
        linear_pred = x_with_const @ res.params.values
        from scipy.stats import norm
        oos_probs.iloc[i] = norm.cdf(linear_pred)
    except Exception as e:
        n_errors += 1
        if not first_error_shown:
            print(f"  First error at i={i} ({model_df.index[i].strftime('%Y-%m')}): {type(e).__name__}: {e}")
            first_error_shown = True
        oos_probs.iloc[i] = float("nan")

    if (i - MIN_WINDOW) % 100 == 0:
        pct = (i - MIN_WINDOW) / total_iters * 100
        print(f"  {pct:5.1f}% complete (month {model_df.index[i].strftime('%Y-%m')})")

# Also generate OOS predictions beyond model_df using the final trained model
# (for dates where features exist but TARGET doesn't)
final_model = sm.Probit(y, sm.add_constant(model_df[oos_features].astype(float))).fit(disp=False, method='bfgs', maxiter=500)
extra_dates = predict_df.index.difference(model_df.index)
if len(extra_dates) > 0:
    oos_probs_full = oos_probs.copy()
    for dt in extra_dates:
        x_curr = predict_df.loc[dt, oos_features].astype(float).values
        x_with_const = np.concatenate([[1.0], x_curr])
        linear_pred = x_with_const @ final_model.params.values
        oos_probs_full[dt] = norm.cdf(linear_pred)
    print(f"Extended OOS predictions by {len(extra_dates)} months to {extra_dates.max().strftime('%Y-%m')}")
else:
    oos_probs_full = oos_probs.copy()

valid_count = oos_probs.notna().sum()
print(f"\nOut-of-sample estimation complete.")
print(f"Valid OOS probabilities: {valid_count} / {total_iters}")
if n_errors > 0:
    print(f"Errors: {n_errors} iterations failed")

## Model Evaluation

Compare all four models using AUROC, Brier Score, AIC, BIC, and Pseudo R².

The BIC-selected model should dominate on BIC by construction. The key
question is whether it also dominates on AUROC out-of-sample.

In [ ]:
# ============================================================
# Evaluation metrics — all models
# ============================================================
y_true = model_df["TARGET"].astype(float)

header = f"{'Model':<28s} {'AUROC':<10s} {'Brier':<10s} {'Pseudo R²':<10s} {'AIC':<10s} {'BIC':<10s} {'# Feat':<6s}"
print(header)
print("-" * len(header))

for name, res_dict in results.items():
    fitted = res_dict["fitted"]
    auroc = roc_auc_score(y_true, fitted)
    brier = brier_score_loss(y_true, fitted)
    pr2 = res_dict["pseudo_r2"]
    aic = res_dict["aic"]
    bic = res_dict["bic"]
    nf = len(res_dict["features"])
    print(f"{name:<28s} {auroc:<10.4f} {brier:<10.4f} {pr2:<10.4f} {aic:<10.1f} {bic:<10.1f} {nf:<6d}")

# OOS metrics for BIC-selected
# Only evaluate OOS on dates where TARGET is known (within model_df)
oos_valid = oos_probs_full.dropna()
oos_eval = oos_valid[oos_valid.index.isin(model_df.index)]
if len(oos_eval) > 0:
    y_oos = model_df.loc[oos_eval.index, "TARGET"].astype(float)
    auroc_oos = roc_auc_score(y_oos, oos_eval)
    brier_oos = brier_score_loss(y_oos, oos_eval)
    print(f"{'BIC-Selected (OOS)':<28s} {auroc_oos:<10.4f} {brier_oos:<10.4f} {'—':<10s} {'—':<10s} {'—':<10s} {len(bic_selected):<6d}")

## Dependent Variable Comparison: Point-in-Time vs. Any-in-Window

The Boston Fed (2020) found "considerable dispersion in predicted recession
probabilities" depending on this choice. Here we fit the BIC-selected model
under **both** definitions and compare.

In [ ]:
# ============================================================
# Compare both dependent variable definitions
# ============================================================
target_point  = data["USREC"].shift(-12)
target_window = data["USREC"].rolling(window=12).max().shift(-12)

dv_results = {}

for dv_name, dv_series in [("Point (t+12)", target_point),
                            ("Window (any in 1-12)", target_window)]:
    # Align with model_df index
    dv = dv_series.reindex(model_df.index).dropna()
    common_idx = dv.index.intersection(model_df.index)
    y_dv = dv.loc[common_idx].astype(float)
    X_dv = sm.add_constant(model_df.loc[common_idx, bic_selected].astype(float))

    try:
        res = sm.Probit(y_dv, X_dv).fit(disp=False, method="bfgs", maxiter=500)
        fitted = res.predict(X_dv)
        auroc = roc_auc_score(y_dv, fitted)
        brier = brier_score_loss(y_dv, fitted)
        dv_results[dv_name] = {
            "fitted": fitted,
            "y": y_dv,
            "pseudo_r2": res.prsquared,
            "auroc": auroc,
            "brier": brier,
        }
        pct_pos = y_dv.mean() * 100
        print(f"{dv_name}:")
        print(f"  Positive rate: {pct_pos:.1f}%   Pseudo R²: {res.prsquared:.4f}   AUROC: {auroc:.4f}   Brier: {brier:.4f}")
    except Exception as e:
        print(f"{dv_name}: FAILED — {e}")

In [ ]:
# ============================================================
# Side-by-side chart: both DV definitions
# ============================================================
if len(dv_results) == 2:
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    usrec = data["USREC"].dropna()

    for ax, (dv_name, dv_dict) in zip(axes, dv_results.items()):
        ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                        color="#d4d4d4", alpha=0.6, label="NBER Recession")
        fitted = dv_dict["fitted"]
        ax.plot(fitted.index, fitted * 100, color="#1f77b4", linewidth=1.2)
        ax.axhline(y=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
        ax.axhline(y=THRESHOLD_WARNING, color="orange", linestyle=":", alpha=0.3, linewidth=0.8)
        ax.set_ylim(0, 100)
        ax.set_ylabel("Probability (%)")
        r2 = dv_dict["pseudo_r2"]
        auroc = dv_dict["auroc"]
        ax.set_title(f"{dv_name}  (Pseudo R²={r2:.4f}, AUROC={auroc:.4f})", fontweight="bold")
        ax.legend(loc="upper right", fontsize=9)
        ax.grid(True, alpha=0.15)

    axes[0].xaxis.set_major_locator(mdates.YearLocator(5))
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.suptitle("Dependent Variable Comparison — BIC-Selected Model", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Could not fit both DV definitions.")

## Stage 5: Visualization

Main chart: 12-month-ahead recession probability with NBER recession shading.
Shows both in-sample (BIC-selected) and out-of-sample probabilities.

In [ ]:
# ============================================================
# Main Recession Probability Chart
# ============================================================
fig, ax = plt.subplots(figsize=(16, 6))

# NBER recession shading
usrec = data["USREC"].dropna()
ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                color="#d4d4d4", alpha=0.6, label="NBER Recession")

# In-sample (BIC-selected)
if "BIC-Selected" in results:
    fitted = results["BIC-Selected"].get("fitted_full", results["BIC-Selected"]["fitted"])
    ax.plot(fitted.index, fitted * 100,
            color="#1f77b4", linewidth=1.2, alpha=0.5,
            label="BIC-Selected Model")

# Out-of-sample
oos_valid = oos_probs_full.dropna()
if len(oos_valid) > 0:
    ax.plot(oos_valid.index, oos_valid * 100,
            color="#d62728", linewidth=1.5,
            label="Out-of-Sample (Expanding Window)")

# Threshold lines
ax.axhline(y=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.3,
           linewidth=0.8, label=f"{THRESHOLD_ELEVATED}% Threshold")
ax.axhline(y=THRESHOLD_WARNING, color="orange", linestyle=":", alpha=0.3,
           linewidth=0.8, label=f"{THRESHOLD_WARNING}% Warning")

# Styling
ax.set_ylim(0, 100)
ax.set_xlim(model_df.index.min(), data.index.max())
ax.set_ylabel("Probability (%)", fontsize=12)
dv_label = "Point-in-Time" if TARGET_DEFINITION == "point" else "Any-in-Window"
ax.set_title(f"U.S. Recession Probability — 12-Month Ahead ({dv_label}, BIC-Selected Probit)",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("recession_probability_main.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved recession_probability_main.png")

## Model Comparison Chart

All four specifications overlaid: NY Fed baseline, Wright, BIC-selected, and full candidate set.

In [ ]:
# ============================================================
# Model comparison chart
# ============================================================
palette = ["#2ca02c", "#ff7f0e", "#1f77b4", "#9467bd"]
fig, ax = plt.subplots(figsize=(16, 6))

usrec = data["USREC"].dropna()
ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                color="#d4d4d4", alpha=0.6, label="NBER Recession")

for (name, res_dict), color in zip(results.items(), palette):
    fitted = res_dict.get("fitted_full", res_dict["fitted"])
    r2 = res_dict["pseudo_r2"]
    bic_val = res_dict["bic"]
    ax.plot(fitted.index, fitted * 100, color=color,
            linewidth=1.0, alpha=0.8,
            label=f"{name} (R²={r2:.3f}, BIC={bic_val:.0f})")

ax.axhline(y=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
ax.set_ylim(0, 100)
ax.set_ylabel("Probability (%)", fontsize=12)
ax.set_title("Model Comparison — 12-Month-Ahead Recession Probability", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("recession_probability_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Current Recession Probability Reading

In [ ]:
# ============================================================
# Current reading — all models (using latest available data)
# ============================================================
print("=" * 65)
print("  CURRENT 12-MONTH-AHEAD RECESSION PROBABILITY")
print("=" * 65)

for name, res_dict in results.items():
    fitted_full = res_dict.get("fitted_full", res_dict["fitted"])
    p = fitted_full.iloc[-1] * 100
    date = fitted_full.index[-1].strftime('%B %Y')
    nf = len(res_dict["features"])
    print(f"  {name:<28s}: {p:5.1f}%   (as of {date}, {nf} features)")

print()

# Latest from BIC-selected
if "BIC-Selected" in results:
    fitted_full = results["BIC-Selected"].get("fitted_full", results["BIC-Selected"]["fitted"])
    latest_prob = fitted_full.iloc[-1] * 100
    latest_date = fitted_full.index[-1]

    if latest_prob > THRESHOLD_ELEVATED:
        print(f"  ELEVATED: BIC-Selected probability ({latest_prob:.1f}%) exceeds {THRESHOLD_ELEVATED}%")
        print(f"  — recession more likely than not within 12 months.")
    elif latest_prob > THRESHOLD_WARNING:
        print(f"  WARNING: BIC-Selected probability ({latest_prob:.1f}%) above {THRESHOLD_WARNING}%")
        print(f"  — warrants attention.")
    else:
        print(f"  LOW: BIC-Selected probability ({latest_prob:.1f}%) below {THRESHOLD_WARNING}%")
        print(f"  — recession risk contained.")

    # Use predict_df for latest indicator values (extends past model_df)
    print("\n  Current indicator readings (BIC-selected features):")
    latest_row = predict_df[bic_selected].iloc[-1]
    for feat in bic_selected:
        cat = feat_to_cat.get(feat, "?")
        print(f"    {feat:<20s} = {latest_row[feat]:>8.2f}   [{cat}]")

## Estrella-Mishkin Quick Estimate

Using pre-estimated parameters from Estrella & Trubin (2006):
**P(recession) = Φ(−0.6045 − 0.7374 × spread)**

No model fitting required — just plug in the current spread.

In [ ]:
# ============================================================
# Estrella-Mishkin closed-form estimate
# ============================================================
spread_col = "SPREAD" if "SPREAD" in data.columns else "T10Y3M" if "T10Y3M" in data.columns else None

if spread_col:
    spread_latest = data[spread_col].dropna().iloc[-1]
    em_prob = stats.norm.cdf(-0.6045 - 0.7374 * spread_latest) * 100

    print(f"Current 10Y-3M spread:      {spread_latest:.2f}%")
    print(f"Estrella-Mishkin 12m prob:   {em_prob:.1f}%")
    print()

    # Plot the mapping curve
    spread_range = np.linspace(-4, 5, 200)
    em_curve = stats.norm.cdf(-0.6045 - 0.7374 * spread_range) * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(spread_range, em_curve, "b-", linewidth=2)
    ax.axvline(x=spread_latest, color="red", linestyle="--", alpha=0.6,
               label=f"Current spread: {spread_latest:.2f}%")
    ax.axvline(x=0, color="gray", linestyle=":", alpha=0.4, label="Inversion point")
    ax.scatter([spread_latest], [em_prob], color="red", s=80, zorder=5)
    ax.annotate(f"{em_prob:.1f}%", (spread_latest, em_prob),
                textcoords="offset points", xytext=(15, 10), fontsize=11,
                arrowprops=dict(arrowstyle="->", color="red"))
    ax.set_xlabel("10Y-3M Treasury Spread (%)", fontsize=12)
    ax.set_ylabel("Recession Probability (%)", fontsize=12)
    ax.set_title("Estrella-Mishkin Yield Curve Model", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.15)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("No spread series available for Estrella-Mishkin estimate.")

## Feature Importance — BIC-Selected Model

z-statistics and marginal effects for the BIC-selected features.
Red bars = statistically significant at 5%; blue = not significant.

In [ ]:
# ============================================================
# Feature importance: z-statistics and marginal effects
# ============================================================
if "BIC-Selected" in results:
    # Re-estimate with newton method to ensure covariance matrix is computed
    bic_feats = results["BIC-Selected"]["features"]
    X_refit = sm.add_constant(model_df[bic_feats].astype(float))
    y_refit = model_df["TARGET"].astype(float)

    try:
        res_refit = sm.Probit(y_refit, X_refit).fit(disp=False, method="newton", maxiter=500)
    except Exception:
        # Fall back to bfgs if newton fails
        res_refit = results["BIC-Selected"]["model"]

    coef_df = pd.DataFrame({
        "Coefficient": res_refit.params,
        "Std Error": res_refit.bse,
        "z-stat": res_refit.tvalues,
        "p-value": res_refit.pvalues,
        "|z-stat|": res_refit.tvalues.abs(),
    }).drop("const", errors="ignore")

    coef_df = coef_df.sort_values("|z-stat|", ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(coef_df) * 0.5)))

    # Panel 1: z-statistics
    colors_z = ["#d62728" if p < 0.05 else "#aec7e8" for p in coef_df["p-value"]]
    axes[0].barh(coef_df.index, coef_df["|z-stat|"], color=colors_z)
    axes[0].axvline(x=1.96, color="gray", linestyle="--", alpha=0.5, label="5% significance")
    axes[0].set_xlabel("|z-statistic|")
    axes[0].set_title("Statistical Significance of Predictors")
    axes[0].legend()

    # Panel 2: Marginal effects at the mean
    try:
        mfx = res_refit.get_margeff(at="mean")
        mfx_vals = mfx.margeff
    except (ValueError, np.linalg.LinAlgError):
        # If covariance still unavailable, compute marginal effects manually:
        # dP/dx = phi(X*beta) * beta  (evaluated at the mean)
        X_mean = X_refit.mean()
        linear_pred = X_mean @ res_refit.params
        phi_val = stats.norm.pdf(linear_pred)
        mfx_vals = (phi_val * res_refit.params).drop("const", errors="ignore").values

    mfx_df = pd.DataFrame({
        "Marginal Effect": mfx_vals,
        "Feature": list(coef_df.index),
    }).set_index("Feature").sort_values("Marginal Effect")

    mfx_colors = ["#d62728" if v > 0 else "#2ca02c" for v in mfx_df["Marginal Effect"]]
    axes[1].barh(mfx_df.index, mfx_df["Marginal Effect"], color=mfx_colors)
    axes[1].axvline(x=0, color="gray", linewidth=0.8)
    axes[1].set_xlabel("Marginal Effect on P(Recession)")
    axes[1].set_title("Marginal Effects at the Mean")

    plt.suptitle("Feature Importance — BIC-Selected Probit",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

## Correlation Matrix — BIC-Selected Features

In [ ]:
# ============================================================
# Correlation heatmap of BIC-selected features
# ============================================================
corr_cols = [f for f in bic_selected if f in model_df.columns]
corr_matrix = model_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(max(6, len(corr_cols)), max(5, len(corr_cols) * 0.8)))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=7,
                color="white" if abs(corr_matrix.iloc[i, j]) > 0.6 else "black")

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("BIC-Selected Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Historical Recession Detection Performance

How well did each model detect historical recessions 12 months in advance?
Checks peak probability in the 6–18 month window prior to each recession start.

In [ ]:
# ============================================================
# Historical recession detection — all models
# ============================================================
usrec_model = model_df["USREC"]
rec_starts = usrec_model[(usrec_model == 1) & (usrec_model.shift(1) == 0)].index

header = f"{'Recession':<14s}"
for name in results:
    short = name.split("(")[0].strip()[:12]
    header += f" {short:>14s}"
print(header)
print("-" * len(header))

for start in rec_starts:
    row = f"  {start.strftime('%Y-%m'):<12s}"
    for name, res_dict in results.items():
        fitted = res_dict["fitted"]
        window_start = start - pd.DateOffset(months=18)
        window_end = start - pd.DateOffset(months=6)
        window_probs = fitted.loc[window_start:window_end]
        if len(window_probs) > 0:
            peak = window_probs.max() * 100
            marker = " *" if peak > THRESHOLD_WARNING else ""
            row += f" {peak:>12.1f}%{marker}"
        else:
            row += f" {'—':>14s}"
    print(row)

print(f"\n  * = peak probability exceeded {THRESHOLD_WARNING}% warning threshold")

## Export Results

Save all probabilities, indicator data, and model metadata to CSV.

In [ ]:
# ============================================================
# Export to CSV
# ============================================================
export_df = pd.DataFrame(index=model_df.index)
export_df["USREC"] = model_df["USREC"]
export_df["TARGET_actual"] = model_df["TARGET"]
export_df["target_definition"] = TARGET_DEFINITION

for name, res_dict in results.items():
    col_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("-","_")
    export_df[f"Prob_{col_name}"] = res_dict["fitted"] * 100

export_df["Prob_OOS_BIC_Selected"] = oos_probs_full * 100

# Add all BIC-selected indicators
for feat in bic_selected:
    if feat in model_df.columns:
        export_df[feat] = model_df[feat]

export_df.to_csv("recession_probabilities.csv")
print(f"Exported {len(export_df)} rows x {len(export_df.columns)} columns to recession_probabilities.csv")
print(f"Columns: {list(export_df.columns)}")
export_df.tail()

## References

1. **Estrella, A. & Mishkin, F.S. (1998)**. "Predicting U.S. Recessions: Financial Variables as Leading Indicators." *Review of Economics and Statistics*, 80(1), 45-61.
2. **Wright, J.H. (2006)**. "The Yield Curve and Predicting Recessions." *Federal Reserve Board FEDS Working Paper* No. 2006-07.
3. **Kauppi, H. & Saikkonen, P. (2008)**. "Predicting U.S. Recessions with Dynamic Binary Response Models." *Review of Economics and Statistics*, 90(4), 777-791.
4. **Berge, T.J. (2014)**. "Predicting Recessions with Leading Indicators: Model Averaging and Links to the Financial Crisis." *Federal Reserve Bank of Kansas City Working Paper*.
5. **Federal Reserve Board FEDS Notes (2018, 2019)**. Various notes on recession probability models.
6. **Sahm, C. (2019)**. "Direct Stimulus Payments to Individuals." *Brookings Institution*.
7. **McCracken, M.W. & Ng, S. (2016)**. "FRED-MD: A Monthly Database for Macroeconomic Research." *Journal of Business & Economic Statistics*, 34(4), 574-589.
8. **Boston Fed (2020)**. On dispersion in recession probabilities from dependent variable construction.
9. **Berge, T.J. & Jordà, Ò. (2011)**. "Evaluating the Classification of Economic Activity into Recessions and Expansions." *American Economic Journal: Macroeconomics*.
10. **Bellego, C. & Ferrara, L. (2009)**. "Forecasting Euro Area Recessions Using Time-Varying Binary Response Models for Financial Variables." *ECB Working Paper*.

---

*Data sourced from FRED (Federal Reserve Bank of St. Louis).*
*Feature selection: forward stepwise by BIC. Dependent variable: configurable.*

## Bootstrap Confidence Intervals

Point estimates are insufficient for decision-making. This section computes
a 90% confidence interval on the current recession probability by resampling
the training data 1,000 times and re-estimating the probit each time.

In [ ]:
# ============================================================
# Bootstrap CI on current recession probability
# ============================================================
N_BOOT = 1000
np.random.seed(42)

boot_probs = []
x_latest = predict_df[bic_selected].iloc[-1].astype(float).values
x_with_const = np.concatenate([[1.0], x_latest])

bic_prob = stats.norm.cdf(x_with_const @ results["BIC-Selected"]["model"].params.values) * 100

print(f"Running {N_BOOT} bootstrap iterations...")

for b in range(N_BOOT):
    idx = np.random.choice(len(model_df), size=len(model_df), replace=True)
    boot_data = model_df.iloc[idx]
    y_b = boot_data["TARGET"].astype(float)
    X_b = sm.add_constant(boot_data[bic_selected].astype(float))
    try:
        res_b = sm.Probit(y_b, X_b).fit(disp=False, method="bfgs", maxiter=200)
        lp = x_with_const @ res_b.params.values
        boot_probs.append(stats.norm.cdf(lp))
    except Exception:
        pass

boot_probs = np.array(boot_probs)
ci_lower = np.percentile(boot_probs, 5) * 100
ci_upper = np.percentile(boot_probs, 95) * 100
ci_median = np.percentile(boot_probs, 50) * 100

print(f"\nBootstrap results ({len(boot_probs)} successful iterations):")
print(f"  Median:     {ci_median:.2f}%")
print(f"  90% CI:     [{ci_lower:.2f}%, {ci_upper:.2f}%]")
print(f"  Point est:  {bic_prob:.2f}%")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(boot_probs * 100, bins=50, color="#1f77b4", alpha=0.7, edgecolor="white")
ax.axvline(x=ci_lower, color="red", linestyle="--", label=f"90% CI: [{ci_lower:.1f}%, {ci_upper:.1f}%]")
ax.axvline(x=ci_upper, color="red", linestyle="--")
ax.axvline(x=bic_prob, color="black", linewidth=2, label=f"Point estimate: {bic_prob:.1f}%")
ax.set_xlabel("Recession Probability (%)")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Distribution of Current Recession Probability", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## Scenario Analysis: Sensitivity to Indicator Movements

How much does the recession probability change if each indicator moves by
one historical standard deviation? Shows which indicators the committee
should watch most closely.

In [ ]:
# ============================================================
# Sensitivity analysis: +/- 1 std dev shock per indicator
# ============================================================
res_bic = results["BIC-Selected"]["model"]
baseline_x = predict_df[bic_selected].iloc[-1].astype(float).values
baseline_xc = np.concatenate([[1.0], baseline_x])
baseline_prob = stats.norm.cdf(baseline_xc @ res_bic.params.values) * 100

print(f"Baseline probability: {baseline_prob:.2f}%\n")
print(f"{"Indicator":<20s} {"Current":>10s} {"1 SD":>8s} {"-1 SD Prob":>12s} {"+1 SD Prob":>12s} {"Impact":>10s}")
print("-" * 72)

scenario_data = []
for j, feat in enumerate(bic_selected):
    sd = model_df[feat].std()
    current_val = baseline_x[j]

    x_down = baseline_x.copy()
    x_down[j] -= sd
    xc_down = np.concatenate([[1.0], x_down])
    prob_down = stats.norm.cdf(xc_down @ res_bic.params.values) * 100

    x_up = baseline_x.copy()
    x_up[j] += sd
    xc_up = np.concatenate([[1.0], x_up])
    prob_up = stats.norm.cdf(xc_up @ res_bic.params.values) * 100

    impact = prob_up - prob_down
    scenario_data.append((feat, current_val, sd, prob_down, prob_up, impact))
    print(f"{feat:<20s} {current_val:>10.2f} {sd:>8.2f} {prob_down:>11.2f}% {prob_up:>11.2f}% {impact:>+9.2f}pp")

# Adverse scenario
x_adverse = baseline_x.copy()
for j, feat in enumerate(bic_selected):
    coef = res_bic.params.iloc[j+1]
    sd = model_df[feat].std()
    if coef > 0:
        x_adverse[j] += sd
    else:
        x_adverse[j] -= sd
xc_adverse = np.concatenate([[1.0], x_adverse])
prob_adverse = stats.norm.cdf(xc_adverse @ res_bic.params.values) * 100

print(f"\nAdverse scenario (all indicators +1 SD toward recession): {prob_adverse:.1f}%")
print(f"Baseline:                                                   {baseline_prob:.1f}%")

fig, ax = plt.subplots(figsize=(10, max(4, len(bic_selected) * 0.6)))
feats = [s[0] for s in scenario_data]
impacts = [s[5] for s in scenario_data]
colors = ["#d62728" if imp > 0 else "#2ca02c" for imp in impacts]
ax.barh(feats, impacts, color=colors)
ax.axvline(x=0, color="gray", linewidth=0.8)
ax.set_xlabel("Impact on Probability (pp) from +1 SD Shock")
ax.set_title("Sensitivity Analysis: Which Indicators Move the Needle?", fontweight="bold")
plt.tight_layout()
plt.show()

## Leave-One-Recession-Out Cross-Validation

With only ~6 recession episodes, this test trains on all recessions except
one, then evaluates whether it would have detected the held-out recession.

In [ ]:
# ============================================================
# Leave-one-recession-out validation
# ============================================================
usrec_model = model_df["USREC"]
rec_starts = usrec_model[(usrec_model == 1) & (usrec_model.shift(1) == 0)].index
rec_ends = usrec_model[(usrec_model == 0) & (usrec_model.shift(1) == 1)].index

episodes = []
for start in rec_starts:
    matching_ends = rec_ends[rec_ends > start]
    end = matching_ends[0] if len(matching_ends) > 0 else model_df.index[-1]
    episodes.append((start, end))

print(f"Found {len(episodes)} recession episodes:\n")
print(f"{"Episode":<6s} {"Start":<12s} {"End":<12s} {"Months":>8s} {"Peak Prob (held out)":>22s} {"Detected?":>12s}")
print("-" * 72)

loro_results = []
for ep_idx, (ep_start, ep_end) in enumerate(episodes):
    holdout_start = ep_start - pd.DateOffset(months=12)
    holdout_end = ep_end
    train_mask = ~((model_df.index >= holdout_start) & (model_df.index <= holdout_end))
    train_data = model_df[train_mask]
    y_train = train_data["TARGET"].astype(float)
    X_train = sm.add_constant(train_data[bic_selected].astype(float))

    try:
        res_loro = sm.Probit(y_train, X_train).fit(disp=False, method="bfgs", maxiter=500)
        pred_start = ep_start - pd.DateOffset(months=18)
        pred_end = ep_start - pd.DateOffset(months=6)
        pred_window = model_df.loc[pred_start:pred_end, bic_selected]
        if len(pred_window) > 0:
            X_pred = sm.add_constant(pred_window.astype(float))
            pred_probs = res_loro.predict(X_pred) * 100
            peak = pred_probs.max()
            detected = peak > THRESHOLD_WARNING
        else:
            peak = float("nan")
            detected = False
        months = (ep_end - ep_start).days // 30
        det_str = "YES" if detected else "no"
        loro_results.append({"start": ep_start, "peak": peak, "detected": detected})
        print(f"  {ep_idx+1:<5d} {ep_start.strftime("%Y-%m"):<12s} {ep_end.strftime("%Y-%m"):<12s} {months:>8d} {peak:>21.1f}% {det_str:>12s}")
    except Exception as e:
        print(f"  {ep_idx+1:<5d} {ep_start.strftime("%Y-%m"):<12s} FAILED: {e}")

n_detected = sum(1 for r in loro_results if r["detected"])
n_total = len(loro_results)
print(f"\nDetection rate (leave-one-out): {n_detected}/{n_total} recessions detected at >{THRESHOLD_WARNING}% threshold")

## Penalized Probit (Firth-Type Bias Reduction)

The standard probit warns about quasi-separation. A penalized likelihood
approach (adding a Jeffreys prior / Firth penalty) regularizes the
coefficients and produces finite, stable estimates even with
quasi-separation. We compare the penalized model to the standard one.

In [ ]:
# ============================================================
# Penalized probit via L2 regularization (practical Firth approx)
# statsmodels does not have native Firth for probit, but we can
# approximate it with sklearn LogisticRegression (L2-penalized
# logit ≈ penalized probit in practice) and compare.
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Standardize features for fair regularization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(model_df[bic_selected].astype(float))
y_pen = model_df["TARGET"].astype(float).values

# Fit penalized logistic (C=1.0 is moderate regularization)
pen_model = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000)
pen_model.fit(X_scaled, y_pen)

# Predict on latest data
x_latest_scaled = scaler.transform(predict_df[bic_selected].iloc[[-1]].astype(float))
pen_prob = pen_model.predict_proba(x_latest_scaled)[0, 1] * 100

# Full fitted probabilities for comparison
X_scaled_full = scaler.transform(predict_df[bic_selected].dropna().astype(float))
pen_fitted_full = pen_model.predict_proba(X_scaled_full)[:, 1]
pen_fitted_series = pd.Series(pen_fitted_full, index=predict_df[bic_selected].dropna().index)

# Compare
bic_latest = results["BIC-Selected"].get("fitted_full", results["BIC-Selected"]["fitted"]).iloc[-1] * 100
print(f"Standard probit (MLE):      {bic_latest:.2f}%")
print(f"Penalized logistic (L2):    {pen_prob:.2f}%")
print(f"Difference:                 {abs(pen_prob - bic_latest):.2f}pp")
print()

if abs(pen_prob - bic_latest) < 5:
    print("The standard and penalized estimates agree closely.")
    print("Quasi-separation is not materially affecting the probability estimate.")
else:
    print("WARNING: Substantial divergence between standard and penalized estimates.")
    print("Quasi-separation may be distorting the standard probit.")

# Store for IC report
penalized_prob = pen_prob

## Model Consensus Analysis

When multiple model specifications agree, confidence is higher.
When they diverge, the committee should weight the range, not just one number.

In [ ]:
# ============================================================
# Model consensus across specifications
# ============================================================
model_probs = {}
for name, res_dict in results.items():
    fitted_full = res_dict.get("fitted_full", res_dict["fitted"])
    model_probs[name] = fitted_full.iloc[-1] * 100

model_probs["Penalized (L2)"] = penalized_prob

# Estrella-Mishkin closed-form
spread_col = "SPREAD" if "SPREAD" in data.columns else "T10Y3M"
if spread_col in data.columns:
    spread_val = data[spread_col].dropna().iloc[-1]
    model_probs["Estrella-Mishkin"] = stats.norm.cdf(-0.6045 - 0.7374 * spread_val) * 100

prob_values = list(model_probs.values())
prob_range = max(prob_values) - min(prob_values)
prob_mean = np.mean(prob_values)
prob_median = np.median(prob_values)

print("Model Consensus Report")
print("=" * 50)
for name, p in sorted(model_probs.items(), key=lambda x: x[1], reverse=True):
    bar = "+" * int(p / 2)  # simple text bar
    print(f"  {name:<28s} {p:5.1f}%  {bar}")

print(f"\n  Range:   {min(prob_values):.1f}% — {max(prob_values):.1f}%  (spread: {prob_range:.1f}pp)")
print(f"  Mean:    {prob_mean:.1f}%")
print(f"  Median:  {prob_median:.1f}%")
print()

if prob_range < 15:
    consensus = "STRONG"
    consensus_msg = "Models agree — high confidence in the estimate range."
elif prob_range < 30:
    consensus = "MODERATE"
    consensus_msg = "Some divergence — consider the full range, not just one model."
else:
    consensus = "WEAK"
    consensus_msg = "Significant divergence — the models disagree. Use the range as bounds."

print(f"  Consensus: {consensus}")
print(f"  {consensus_msg}")

# Visual
fig, ax = plt.subplots(figsize=(10, max(3, len(model_probs) * 0.5)))
names = list(model_probs.keys())
vals = list(model_probs.values())
colors = ["#1f77b4" if n == "BIC-Selected" else "#aec7e8" for n in names]
ax.barh(names, vals, color=colors)
ax.axvline(x=THRESHOLD_WARNING, color="orange", linestyle=":", alpha=0.6, label=f"{THRESHOLD_WARNING}% warning")
ax.axvline(x=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.4, label=f"{THRESHOLD_ELEVATED}% elevated")
ax.set_xlabel("Recession Probability (%)")
ax.set_title(f"Model Consensus: {consensus}", fontweight="bold")
ax.legend(loc="upper right")
ax.set_xlim(0, max(max(vals) * 1.3, THRESHOLD_WARNING + 5))
plt.tight_layout()
plt.show()

---
---

# Investment Committee Report: U.S. Recession Probability

*Generated from a 12-month-ahead probit model using Federal Reserve economic data.*

In [ ]:
# ============================================================
# IC REPORT — PAGE 1: Executive Dashboard
# ============================================================
from datetime import datetime

report_date = predict_df.index[-1].strftime("%B %Y")
run_date = datetime.now().strftime("%B %d, %Y")

# Gather probabilities from all models
ic_probs = {}
for name, res_dict in results.items():
    fitted_full = res_dict.get("fitted_full", res_dict["fitted"])
    ic_probs[name] = fitted_full.iloc[-1] * 100

bic_prob = ic_probs.get("BIC-Selected", 0)

# --- Build the dashboard figure ---
fig = plt.figure(figsize=(16, 20))
fig.patch.set_facecolor("white")

# Title band
fig.text(0.5, 0.97, "U.S. RECESSION PROBABILITY — 12-MONTH OUTLOOK",
         ha="center", va="top", fontsize=18, fontweight="bold", color="#1a1a2e")
fig.text(0.5, 0.955, f"Data through {report_date}  |  Report generated {run_date}",
         ha="center", va="top", fontsize=11, color="#555555")

# ------------------------------------------------------------------
# Panel 1: Headline probability gauge (top)
# ------------------------------------------------------------------
ax_gauge = fig.add_axes([0.1, 0.82, 0.8, 0.12])
ax_gauge.set_xlim(0, 100)
ax_gauge.set_ylim(0, 1)

# Background gradient bar
for x in range(100):
    if x < THRESHOLD_WARNING:
        color = "#2ecc71"  # green
    elif x < THRESHOLD_ELEVATED:
        color = "#f39c12"  # orange
    else:
        color = "#e74c3c"  # red
    ax_gauge.axvspan(x, x+1, alpha=0.3, color=color, lw=0)

# Marker for current probability
ax_gauge.axvline(x=bic_prob, color="#1a1a2e", linewidth=3, zorder=5)
ax_gauge.plot(bic_prob, 0.5, "v", color="#1a1a2e", markersize=15, zorder=5)

# Labels
ax_gauge.text(THRESHOLD_WARNING/2, -0.3, "LOW", ha="center", fontsize=10, color="#2ecc71", fontweight="bold")
ax_gauge.text((THRESHOLD_WARNING + THRESHOLD_ELEVATED)/2, -0.3, "ELEVATED", ha="center", fontsize=10, color="#f39c12", fontweight="bold")
ax_gauge.text((THRESHOLD_ELEVATED + 100)/2, -0.3, "HIGH", ha="center", fontsize=10, color="#e74c3c", fontweight="bold")

ax_gauge.text(bic_prob, 1.15, f"{bic_prob:.1f}%", ha="center", fontsize=22, fontweight="bold", color="#1a1a2e")
ax_gauge.text(bic_prob, 1.55, "BIC-Selected Model", ha="center", fontsize=10, color="#555555")

ax_gauge.axvline(x=THRESHOLD_WARNING, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax_gauge.axvline(x=THRESHOLD_ELEVATED, color="gray", linewidth=0.8, linestyle="--", alpha=0.5)
ax_gauge.set_yticks([])
ax_gauge.set_xticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
ax_gauge.set_xlabel("Probability (%)", fontsize=10)
ax_gauge.spines["top"].set_visible(False)
ax_gauge.spines["left"].set_visible(False)
ax_gauge.spines["right"].set_visible(False)

# ------------------------------------------------------------------
# Panel 2: Model comparison table
# ------------------------------------------------------------------
ax_table = fig.add_axes([0.08, 0.72, 0.84, 0.08])
ax_table.axis("off")

table_data = []
# Exclude Full Candidate Set — overfits with 30 features on ~60 recession obs
ic_models = {k: v for k, v in results.items() if k != "Full Candidate Set"}
for name, res_dict in ic_models.items():
    fitted_full = res_dict.get("fitted_full", res_dict["fitted"])
    p = fitted_full.iloc[-1] * 100
    r2 = res_dict["pseudo_r2"]
    nf = len(res_dict["features"])
    bic_val = res_dict["bic"]
    signal = "LOW" if p < THRESHOLD_WARNING else ("ELEVATED" if p < THRESHOLD_ELEVATED else "HIGH")
    table_data.append([name, f"{p:.1f}%", signal, f"{r2:.3f}", f"{bic_val:.0f}", str(nf)])

col_labels = ["Model", "Probability", "Signal", "Pseudo R²", "BIC", "Features"]
table = ax_table.table(cellText=table_data, colLabels=col_labels, loc="center",
                       cellLoc="center", colWidths=[0.25, 0.12, 0.1, 0.12, 0.1, 0.1])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)

# Style header row
for j in range(len(col_labels)):
    table[0, j].set_facecolor("#1a1a2e")
    table[0, j].set_text_props(color="white", fontweight="bold")

# Color the signal column
for row_idx in range(len(table_data)):
    signal = table_data[row_idx][2]
    cell = table[row_idx + 1, 2]
    if signal == "LOW":
        cell.set_facecolor("#d4edda")
    elif signal == "ELEVATED":
        cell.set_facecolor("#fff3cd")
    else:
        cell.set_facecolor("#f8d7da")

# ------------------------------------------------------------------
# Panel 3: Historical probability chart
# ------------------------------------------------------------------
ax_hist = fig.add_axes([0.08, 0.42, 0.84, 0.27])

usrec = data["USREC"].dropna()
ax_hist.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                     color="#e0e0e0", alpha=0.7, label="NBER Recession")

if "BIC-Selected" in results:
    fitted = results["BIC-Selected"].get("fitted_full", results["BIC-Selected"]["fitted"])
    ax_hist.plot(fitted.index, fitted * 100, color="#1a1a2e", linewidth=1.3,
                 label="BIC-Selected Model")

oos_valid = oos_probs_full.dropna()
if len(oos_valid) > 0:
    ax_hist.plot(oos_valid.index, oos_valid * 100, color="#e74c3c", linewidth=1.3,
                 alpha=0.8, label="Out-of-Sample (Expanding Window)")

ax_hist.axhline(y=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
ax_hist.axhline(y=THRESHOLD_WARNING, color="orange", linestyle=":", alpha=0.3, linewidth=0.8)

ax_hist.set_ylim(0, 100)
ax_hist.set_ylabel("Probability (%)", fontsize=11)
ax_hist.set_title("Historical 12-Month-Ahead Recession Probability", fontsize=13, fontweight="bold", pad=10)
ax_hist.legend(loc="upper right", fontsize=9)
ax_hist.xaxis.set_major_locator(mdates.YearLocator(5))
ax_hist.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_hist.grid(True, alpha=0.12)

# ------------------------------------------------------------------
# Panel 4: Current indicator readings
# ------------------------------------------------------------------
ax_ind = fig.add_axes([0.08, 0.26, 0.84, 0.12])
ax_ind.axis("off")

ind_data = []
for feat in bic_selected:
    val = predict_df[feat].iloc[-1]
    cat = feat_to_cat.get(feat, "")
    # Historical context: percentile rank
    pctile = (data[feat].dropna() < val).mean() * 100
    ind_data.append([feat, cat, f"{val:.2f}", f"{pctile:.0f}th"])

ind_labels = ["Indicator", "Category", "Current Value", "Historical Percentile"]
ind_table = ax_ind.table(cellText=ind_data, colLabels=ind_labels, loc="center",
                         cellLoc="center", colWidths=[0.22, 0.22, 0.18, 0.2])
ind_table.auto_set_font_size(False)
ind_table.set_fontsize(10)
ind_table.scale(1, 1.5)

for j in range(len(ind_labels)):
    ind_table[0, j].set_facecolor("#1a1a2e")
    ind_table[0, j].set_text_props(color="white", fontweight="bold")

ax_ind.set_title("Current Indicator Readings (BIC-Selected Features)", fontsize=12,
                 fontweight="bold", pad=15)

# ------------------------------------------------------------------
# Panel 5: Interpretation text
# ------------------------------------------------------------------
ax_text = fig.add_axes([0.08, 0.02, 0.84, 0.22])
ax_text.axis("off")

if bic_prob < THRESHOLD_WARNING:
    assessment = (
        f"The BIC-selected probit model estimates a {bic_prob:.1f}% probability of recession "
        f"within the next 12 months (90% CI: [{ci_lower:.1f}%, {ci_upper:.1f}%]), well below the "
        f"{THRESHOLD_WARNING}% warning threshold. Model consensus is {consensus}: the {len(model_probs)} "
        f"specifications tested range from {min(prob_values):.1f}% to {max(prob_values):.1f}%. "
        f"The penalized estimate ({penalized_prob:.1f}%) confirms that quasi-separation is not "
        f"distorting the result. Overall, the model does not signal elevated recession risk at this time."
    )
elif bic_prob < THRESHOLD_ELEVATED:
    assessment = (
        f"The BIC-selected probit model estimates a {bic_prob:.1f}% probability of recession "
        f"within the next 12 months (90% CI: [{ci_lower:.1f}%, {ci_upper:.1f}%]), above the "
        f"{THRESHOLD_WARNING}% warning threshold. Model consensus is {consensus}: specifications "
        f"range from {min(prob_values):.1f}% to {max(prob_values):.1f}%. The committee should "
        f"monitor the underlying indicators for further deterioration."
    )
else:
    assessment = (
        f"The BIC-selected probit model estimates a {bic_prob:.1f}% probability of recession "
        f"within the next 12 months (90% CI: [{ci_lower:.1f}%, {ci_upper:.1f}%]), exceeding the "
        f"{THRESHOLD_ELEVATED}% threshold. Model consensus is {consensus}: specifications range "
        f"from {min(prob_values):.1f}% to {max(prob_values):.1f}%. The committee should consider "
        f"defensive positioning and closely monitor all indicator categories."
    )

ax_text.text(0, 0.95, "Assessment", fontsize=13, fontweight="bold", color="#1a1a2e",
             transform=ax_text.transAxes, va="top")
ax_text.text(0, 0.78, assessment, fontsize=10.5, color="#333333",
             transform=ax_text.transAxes, va="top", wrap=True,
             linespacing=1.5,
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#f8f9fa", edgecolor="#dee2e6"))

dv_label = "point-in-time (recession at month t+12)" if TARGET_DEFINITION == "point" else "any-in-window (any recession in months t+1 to t+12)"
ax_text.text(0, 0.22, "Model specification", fontsize=10, fontweight="bold", color="#1a1a2e",
             transform=ax_text.transAxes, va="top")
ax_text.text(0, 0.1,
             f"Probit regression  |  Target: {dv_label}  |  "
             f"Training: {model_df.index.min().strftime('%Y-%m')} to {model_df.index.max().strftime('%Y-%m')}  |  "
             f"{len(model_df)} obs  |  Features selected by BIC",
             fontsize=9, color="#777777", transform=ax_text.transAxes, va="top")

plt.savefig("ic_report_page1.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved ic_report_page1.png")

---

# Appendix: Methodology & Mathematical Framework

*For investment committee review — documents the statistical basis, variable selection,
and computation of the recession probability estimate.*

In [ ]:
# ============================================================
# IC REPORT — PAGE 2: Methodology & Math
# ============================================================

fig = plt.figure(figsize=(16, 22))
fig.patch.set_facecolor("white")

fig.text(0.5, 0.98, "METHODOLOGY & MATHEMATICAL FRAMEWORK",
         ha="center", va="top", fontsize=18, fontweight="bold", color="#1a1a2e")
fig.text(0.5, 0.965, "Appendix to U.S. Recession Probability Report",
         ha="center", va="top", fontsize=11, color="#555555")

# Use a single axis for text layout
ax = fig.add_axes([0.06, 0.02, 0.88, 0.92])
ax.axis("off")

y = 1.0  # current y position (top = 1.0, bottom = 0.0)

def section(title, y_pos):
    ax.text(0, y_pos, title, fontsize=14, fontweight="bold", color="#1a1a2e",
            transform=ax.transAxes, va="top")
    return y_pos - 0.025

def body(text, y_pos, indent=0.02):
    ax.text(indent, y_pos, text, fontsize=10, color="#333333",
            transform=ax.transAxes, va="top", wrap=True, linespacing=1.6,
            fontfamily="monospace" if "=" in text and ("+" in text or "\u03a6" in text) else "sans-serif")
    lines = text.count("\n") + 1
    return y_pos - (0.018 * lines + 0.01)

def formula(text, y_pos):
    ax.text(0.5, y_pos, text, fontsize=12, color="#1a1a2e",
            transform=ax.transAxes, va="top", ha="center",
            fontfamily="serif", fontstyle="italic",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f4f8", edgecolor="#c0c8d4"))
    return y_pos - 0.045

# --- Section 1: The Probit Model ---
y = section("1. The Probit Model", y)
y = body(
    "The model estimates the conditional probability of a U.S. recession occurring\n"
    "12 months in the future using a probit regression — a generalized linear model\n"
    "that maps a linear combination of predictors through the standard normal CDF\n"
    "to produce a probability bounded between 0 and 1.",
    y)

y = formula("P(Recession\u209c\u208A\u2081\u2082 = 1 | X\u209c) = \u03a6(\u03b1\u2080 + \u03b1\u2081X\u2081\u209c + \u03b1\u2082X\u2082\u209c + ... + \u03b1\u2096X\u2096\u209c)", y)

y = body(
    "where \u03a6(\u00b7) is the standard normal cumulative distribution function,\n"
    "X\u209c is the vector of economic indicators observed at time t, and the\n"
    "\u03b1 coefficients are estimated by maximum likelihood (MLE).",
    y)

# --- Section 2: Why Probit ---
y = section("2. Why Probit Regression?", y)
y = body(
    "\u2022 Estrella & Mishkin (1998) established probit as the standard for recession forecasting\n"
    "\u2022 The normal CDF naturally bounds output to [0, 1] without ad hoc truncation\n"
    "\u2022 Connects to a latent variable interpretation: an unobserved 'economic health'\n"
    "  index crosses a threshold during recessions\n"
    "\u2022 Provides coefficient p-values, confidence intervals, and pseudo R\u00b2 for inference\n"
    "\u2022 Used by NY Fed, Cleveland Fed, and Federal Reserve Board in published models",
    y)

# --- Section 3: Variable Selection ---
y = section("3. Data-Driven Variable Selection", y)
y = body(
    "Features are selected by forward stepwise BIC (Bayesian Information Criterion),\n"
    "not hardcoded. BIC balances goodness-of-fit against model complexity:",
    y)

y = formula("BIC = -2 \u00b7 ln(L) + k \u00b7 ln(n)", y)

y = body(
    "where L = maximized likelihood, k = number of parameters, n = sample size.\n"
    "The ln(n) penalty is stronger than AIC's penalty of 2k, favoring parsimony.\n"
    "This is consistent with Berge (2014): at the 12-month horizon, simpler models\n"
    "outperform complex ones out of sample.",
    y)

# --- Section 4: Actual coefficients ---
y = section("4. Estimated Model (BIC-Selected)", y)

if "BIC-Selected" in results:
    res = results["BIC-Selected"]["model"]
    coef_lines = f"Intercept (\u03b1\u2080) = {res.params.iloc[0]:>10.4f}\n"
    for j, feat in enumerate(bic_selected):
        coef = res.params.iloc[j+1]
        pval = res.pvalues.iloc[j+1]
        sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
        coef_lines += f"\u03b1({feat}) = {coef:>10.4f}   (p = {pval:.4f}) {sig}\n"
    y = body(coef_lines, y, indent=0.05)

y = body("Significance: *** p<0.001  ** p<0.01  * p<0.05", y, indent=0.05)

# --- Section 5: Computing the probability ---
y = section("5. Computing the Probability (Worked Example)", y)

if "BIC-Selected" in results:
    res = results["BIC-Selected"]["model"]
    latest = predict_df[bic_selected].iloc[-1]
    
    # Build the linear predictor string
    terms = [f"{res.params.iloc[0]:.4f}"]
    for j, feat in enumerate(bic_selected):
        coef = res.params.iloc[j+1]
        val = latest[feat]
        terms.append(f"({coef:+.4f} \u00d7 {val:.2f})")
    
    linear_pred = res.params.iloc[0] + sum(res.params.iloc[j+1] * latest[bic_selected[j]] for j in range(len(bic_selected)))
    final_prob = stats.norm.cdf(linear_pred) * 100
    
    y = body("Step 1: Compute the linear predictor (z-score):", y)
    y = body(f"  z = \u03b1\u2080 + \u03b1\u2081X\u2081 + ... + \u03b1\u2096X\u2096", y, indent=0.05)
    y = body(f"  z = {' '.join(terms[:4])}", y, indent=0.05)
    if len(terms) > 4:
        y = body(f"      {' '.join(terms[4:])}", y, indent=0.05)
    y = body(f"  z = {linear_pred:.4f}", y, indent=0.05)
    
    y = body("Step 2: Pass through the standard normal CDF:", y)
    y = formula(f"P(Recession) = \u03a6({linear_pred:.4f}) = {final_prob:.2f}%", y)

# --- Section 6: Out-of-sample methodology ---
y = section("6. Out-of-Sample Validation", y)
y = body(
    "To prevent lookahead bias, probabilities are also estimated using an\n"
    "expanding-window procedure:\n"
    "\n"
    "  1. Train the probit on data from the start through month t (min 120 months)\n"
    "  2. Generate a probability for month t+12 using only data available at t\n"
    "  3. Advance by one month and repeat\n"
    "\n"
    "This produces honest out-of-sample probabilities where each estimate\n"
    "uses only information a real-time practitioner would have had.",
    y)

# --- Section 7: Key references ---
y = section("7. Academic Foundation", y)
y = body(
    "\u2022 Estrella & Mishkin (1998) — foundational probit framework; yield curve as predictor\n"
    "\u2022 Wright (2006) — adding fed funds rate level improves the spread-only model\n"
    "\u2022 Berge (2014) — at 12-month horizon, parsimony dominates complex models\n"
    "\u2022 Kauppi & Saikkonen (2008) — dynamic probit extensions\n"
    "\u2022 Fed Board FEDS Notes (2018-19) — comparative model evaluation",
    y)

# --- Section 8: Model Limitations ---
y = section("8. Known Limitations", y)
y = body(
    "\u2022 Trained on ~6 independent recession episodes — limited sample for generalization\n"
    "\u2022 Assumes future recessions will resemble historical patterns — novel mechanisms\n"
    "  (pandemics, financial crises, geopolitical shocks) may not be captured\n"
    "\u2022 12-month horizon is long — conditions can change materially within the window\n"
    "\u2022 FRED data has publication lags (1-3 months) — the latest reading may reflect\n"
    "  conditions from 1-2 months ago\n"
    "\u2022 Feature selection (BIC) is in-sample — the optimal feature set may differ\n"
    "  in future regimes\n"
    "\u2022 The model should be one input among many — not a sole basis for allocation",
    y)

# --- Section 9: Robustness checks performed ---
y = section("9. Robustness Checks Performed", y)

loro_rate = f"{n_detected}/{n_total}" if "n_detected" in dir() else "N/A"
y = body(
    f"\u2022 Bootstrap 90% confidence interval: [{ci_lower:.1f}%, {ci_upper:.1f}%]\n"
    f"\u2022 Penalized estimate (L2 regularization): {penalized_prob:.1f}%\n"
    f"\u2022 Model consensus ({len(model_probs)} specifications): {consensus}\n"
    f"\u2022 Leave-one-recession-out detection rate: {loro_rate}\n"
    f"\u2022 Sensitivity analysis: scenario table shows indicator-level impact",
    y)

plt.savefig("ic_report_page2_methodology.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Saved ic_report_page2_methodology.png")